<a href="https://colab.research.google.com/github/nguyenthilananh09081999-beep/-0.6/blob/main/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10주차 실습 (Lab 2): 타이머 기능 추가하기

**[실습 목표]**
Lab 1에서 만든 퀴즈 봇을 발전시켜, **15초 안에** 정답을 맞춰야만 인정되는 '타임어택(Time Attack)' 기능을 추가합니다.
Python의 `time` 모듈을 활용하여 경과 시간을 계산하는 방법을 익히고, 논리 연산자(`and`)를 사용해 두 가지 조건(정답 일치 여부, 시간 내 입력 여부)을 동시에 확인해 봅니다.

---
### 💡 구글 코랩(Colab) 실행 방법 복습
* 코랩은 **'셀(Cell)'** 단위로 코드를 작성하고 실행합니다. 아래에 있는 회색 박스들이 바로 코드를 입력하는 '코드 셀'입니다.
* 코드를 실행하려면 회색 코드 셀을 마우스로 클릭하고 **`Shift + Enter`**를 누르거나, 셀 왼쪽 상단의 **실행(▶️) 버튼**을 클릭합니다.

---
## Step 1: 내 단어장 파일(.csv) 업로드 확인

우리가 만든 파이썬 프로그램이 단어장 파일을 읽어올 수 있도록, 먼저 파일을 코랩에 올려주어야 합니다.

1. 화면 왼쪽 가장자리에 있는 **폴더 모양 아이콘(📁)**을 클릭합니다.
2. `my_words.csv` 파일이 목록에 잘 보이는지 확인하세요.
3. 만약 파일이 없다면, 내 컴퓨터(바탕화면 등)에 있는 `my_words.csv` 파일을 마우스로 끌어서 왼쪽 폴더 창 안에 놓아주세요(드래그 앤 드롭).

---
## Step 2: 파이썬의 '도구함(모듈)' 가져오기

파이썬에는 모든 기능이 처음부터 다 켜져 있지 않습니다. 요리를 할 때 칼과 도마를 꺼내오듯, 코딩할 때도 필요한 기능이 담긴 **'도구함(모듈)'**을 먼저 가져와야(import) 합니다.

* `import csv`: 엑셀이나 콤마(,)로 구분된 CSV 파일을 읽고 쓸 수 있는 도구함을 가져옵니다.
* `import random`: 제비뽑기처럼 무언가를 무작위로(랜덤하게) 골라주는 도구함을 가져옵니다.
* `import time`: 시간을 측정하거나 기록할 수 있는 **'초시계 도구함'**을 가져옵니다. (이번 Lab 2에서 추가됨)

아래 코드 셀을 클릭하고 **`Shift + Enter`**를 눌러 도구함들을 먼저 준비해 주세요.

In [1]:
import csv      # CSV 파일 처리 도구함 가져오기
import random   # 무작위 추출 도구함 가져오기
import time     # 시간 측정을 위한 초시계 도구함 가져오기


## Step 3: 단어장 파일 읽어오기

이제 앞서 업로드한 `my_words.csv` 파일에서 단어를 읽어와 'words'라는 이름의 빈 바구니(리스트)에 담을 차례입니다.
아래 셀을 클릭하고 **`Shift + Enter`**를 눌러 실행해서 성공 메시지가 나오는지 꼭 확인하세요!

In [2]:
filename = "my_words.csv"  # 우리가 읽어올 파일의 이름입니다.
words = []                 # 단어들을 차곡차곡 담을 '빈 바구니(리스트)'를 하나 만듭니다.

try:
    # 파일을 열어서(open) f 라는 별명을 붙여줍니다. (한글이 깨지지 않게 utf-8-sig 사용)
    with open(filename, 'r', encoding='utf-8-sig') as f:
        # csv 도구함을 써서 파일을 한 줄씩 읽을 준비를 합니다.
        reader = csv.reader(f)

        # 파일의 처음부터 끝까지 한 줄씩 반복해서(for) 읽어옵니다.
        for row in reader:
            # 빈 줄이 아니라면 (데이터가 2개 이상 있다면)
            if len(row) >= 2:
                # '한국어'와 '뜻'이라는 이름표를 붙여서 짝을 지어(딕셔너리) words 바구니에 담습니다.
                # strip()은 글자 앞뒤의 쓸데없는 띄어쓰기(공백)를 지워주는 역할을 합니다.
                words.append({'한국어': row[0].strip(), '뜻': row[1].strip()})

    # 파일 읽기가 끝나면 성공 메시지와 함께 총 몇 개의 단어를 담았는지 화면에 알려줍니다.
    print(f"🎉 성공! 총 {len(words)}개의 단어를 불러왔습니다.")
except FileNotFoundError:
    # 만약 파일 이름이 틀렸거나 업로드되지 않았다면 컴퓨터가 멈추는 대신 이 에러 메시지를 띄웁니다.
    print(f"⚠️ 에러: '{filename}' 파일을 찾을 수 없습니다. 왼쪽 폴더에 파일이 잘 업로드되었는지 다시 확인해 주세요!")


🎉 성공! 총 50개의 단어를 불러왔습니다.


## Step 4: 퀴즈 내기 및 15초 시간 재기 (Prompting)

이제 바구니에 담긴 단어들을 무작위로 뽑아서 퀴즈를 냅니다.
이 부분은 사용자가 '종료'라고 입력할 때까지 무한히 퀴즈를 내는 **핵심 로직**입니다.

여기서 가장 중요한 것은 **시간 재기**입니다!
1. 문제가 나오자마자 초시계 버튼을 누르고 (`start_time`)
2. 정답을 입력하면 다시 초시계 버튼을 눌러서 (`end_time`)
3. 두 시간의 차이(`end_time - start_time`)를 계산합니다.

---
### ✨ 코랩 내장 Gemini AI 적극 활용하기!
아래 코드에는 군데군데 **빈칸(`[학생 작성란]`)**이 뚫려 있습니다. 이 빈칸을 직접 완성해야 코드가 실행됩니다.
어떻게 코드를 짜야 할지 막막하다면, 혼자 끙끙대지 말고 **코랩 우측 상단에 있는 반짝이는 별 모양(✨) Gemini 버튼**을 눌러 질문해 보세요!

> **🤖 AI에게 물어볼 프롬프트(질문) 예시**
> * "파이썬에서 현재 시간을 초 단위로 측정하려면 어떤 코드를 써야 해?"
> * "파이썬에서 두 개의 조건(정답이 같고, 시간이 15초 이하)을 `and`로 묶어서 `if`문을 만들고 싶은데 예시를 보여줘."

---
아래 빈칸을 모두 채운 뒤, **`Shift + Enter`**를 눌러 직접 퀴즈를 풀어보세요!
만약 퀴즈를 그만 풀고 싶다면 정답 입력칸에 **종료**라고 치면 됩니다.

In [5]:
if len(words) > 0:
    while True: # 무한히 반복해서 퀴즈를 냅니다.

        # 1. 문제 출제하기
        # random 도구함을 써서 단어 바구니 중 하나를 무작위로 뽑습니다.
        question = random.choice(words)
        kor_word = question['한국어'] # 뽑힌 문제의 외국어/질문 부분
        meaning = question['뜻']    # 뽑힌 문제의 정답 부분

        # 화면에 문제와 제한 시간을 예쁘게 출력해줍니다.
        print(f"\n📝 문제: '{kor_word}' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)")

        # --------------------------------------------------------
        # 2. 타이머 시작! (초시계 켜기)
        # [학생 작성란] 문제가 화면에 뜬 바로 이 순간의 현재 시간을 측정하여 start_time 변수에 저장하세요.
        # --------------------------------------------------------
        start_time = time.time()

        # --------------------------------------------------------
        # 3. 사용자 입력 받기
        # [학생 작성란] 사용자가 키보드로 정답을 입력하고 엔터를 칠 때까지 기다리는 코드를 완성하세요.
        # --------------------------------------------------------
        user_answer = input("정답을 입력하세요: ")

        # --------------------------------------------------------
        # 4. 타이머 종료! (초시계 끄고 걸린 시간 계산하기)
        # [학생 작성란] 사용자가 엔터를 친 직후의 시간을 end_time에 저장하고,
        # 두 시간의 차이를 계산하여 elapsed_time 변수에 저장하세요.
        # --------------------------------------------------------
        end_time = time.time()
        elapsed_time = end_time - start_time

        # --------------------------------------------------------
        # 5. 정답 확인 및 종료 처리
        # --------------------------------------------------------
        # 만약 사용자가 '종료'라고 쳤다면 while 무한 루프를 빠져나옵니다(break).
        if user_answer == '종료':
            print("퀴즈를 마칩니다. 수고하셨습니다!")
            break

        # [학생 작성란] 입력된 값과 정답이 완전히 같은지(==), 그리고 걸린 시간이 15초 이하(<=)인지
        # 두 가지 조건을 'and' 기호를 사용해 동시에 확인하는 if문을 완성하세요.
        if user_answer == meaning and elapsed_time <= 15:
            print(f"✅ 정답입니다! 멋져요! (소요 시간: {elapsed_time:.1f}초)")
        elif user_answer == meaning:
            # 정답은 맞췄지만 15초를 넘긴 경우입니다.
            print(f"⏰ 시간 초과입니다! 정답은 맞췄지만 너무 늦었어요. (소요 시간: {elapsed_time:.1f}초)")
        else:
            print(f"❌ 틀렸습니다. 정답은 '{meaning}' 입니다.")
else:
    print("단어장 데이터가 없습니다. 위의 Step 3 셀을 먼저 실행해서 단어를 불러와주세요.")


📝 문제: '이것' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: cái gì
❌ 틀렸습니다. 정답은 'Cái này' 입니다.

📝 문제: '네/응' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: Vâng
❌ 틀렸습니다. 정답은 'Vâng/Ừ' 입니다.

📝 문제: '안녕하세요' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: Xin chào
✅ 정답입니다! 멋져요! (소요 시간: 7.0초)

📝 문제: '네/응' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: Vâng/Ừ
⏰ 시간 초과입니다! 정답은 맞췄지만 너무 늦었어요. (소요 시간: 17.8초)

📝 문제: '뭐' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: Cái gì
✅ 정답입니다! 멋져요! (소요 시간: 13.7초)

📝 문제: '네/응' 의 뜻은 무엇일까요? (⏳ 제한시간 15초!)
정답을 입력하세요: 종료
퀴즈를 마칩니다. 수고하셨습니다!


## Step 5: 에러 해결 및 개선 (Troubleshooting)

코드를 작성하고 실행(`Shift + Enter`)했는데 빨간 글씨(에러)가 쏟아진다고 당황하지 마세요! 프로그래밍에서 에러는 숨쉬는 것만큼 자연스러운 일입니다.

에러 메시지를 **마우스로 드래그해서 그대로 복사**한 뒤, 코랩 우측 상단의 **Gemini**에게 물어보세요.

> **🤖 질문 예시:**
> "코드를 실행했더니 이런 에러가 났어. 내가 짠 빈칸 코드 부분에 무슨 문제가 있는 거야?"
> `(여기에 복사한 에러 메시지를 붙여넣기)`

타이머 기능이 잘 작동하나요? 시간이 너무 길다면 파이썬 코드에서 제한 시간을 5초(5.0)로 수정하고 다시 `Shift + Enter`를 눌러보세요!